# **04_Feature_Engineering.ipynb**

Objective:
To prepare cleaned datasets for modeling and create suitable training and test partitions.

Method:
Elliptic utilizes a chronological split at time step 34, whereas Ethereum employs an 80/20 stratified random split.

Inputs:


*   elliptic_clean.csv
*   ethereum_clean.csv

Outputs:


*   elliptic_train.csv
*   elliptic_test.csv
*   ethereum_train.csv
*   ethereum_test.csv






In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [23]:
# ELLIPTIC -- chronological split
ell = pd.read_csv("../cleaned_data/elliptic_clean.csv")

In [24]:
CUTOFF = 34 
train = ell[ell["feat_1"] <= CUTOFF].copy()
test = ell[ell["feat_1"] > CUTOFF].copy()

In [25]:

print(f"Elliptic train (steps 1-{CUTOFF}): {train.shape}, illicit rate: {train['label'].mean():.3f}")
print(f"Elliptic test  (steps {CUTOFF+1}-49): {test.shape}, illicit rate: {test['label'].mean():.3f}")

Elliptic train (steps 1-34): (29894, 168), illicit rate: 0.116
Elliptic test  (steps 35-49): (16670, 168), illicit rate: 0.065


In [26]:
# ETHEREUM -- stratified random split
eth = pd.read_csv("../cleaned_data/ethereum_clean.csv")
eth_train, eth_test = train_test_split(
    eth, test_size=0.2, random_state=42, stratify=eth["FLAG"]
)

In [27]:
# Sanity check: confirm no address appears in both splits
overlap = set(eth_train["Address"]) & set(eth_test["Address"])
assert len(overlap) == 0, f"Leakage: {len(overlap)} addresses appear in both train and test"

In [28]:
print(f"\nEthereum train: {eth_train.shape}, fraud rate: {eth_train['FLAG'].mean():.3f}")
print(f"Ethereum test:  {eth_test.shape}, fraud rate: {eth_test['FLAG'].mean():.3f}")
print("Train/test address overlap:", len(overlap))


Ethereum train: (7852, 48), fraud rate: 0.222
Ethereum test:  (1964, 48), fraud rate: 0.222
Train/test address overlap: 0


In [29]:
# Save train/test datasets 

TRAIN_TEST_DATA_PATH = "../train_test_data"

import os
os.makedirs(TRAIN_TEST_DATA_PATH, exist_ok=True)

train.to_csv(
    os.path.join(TRAIN_TEST_DATA_PATH, "elliptic_train.csv"),
    index=False
)

test.to_csv(
    os.path.join(TRAIN_TEST_DATA_PATH, "elliptic_test.csv"),
    index=False
)

eth_train.to_csv(
    os.path.join(TRAIN_TEST_DATA_PATH, "ethereum_train.csv"),
    index=False
)

eth_test.to_csv(
    os.path.join(TRAIN_TEST_DATA_PATH, "ethereum_test.csv"),
    index=False
)

print("All four train/test files saved successfully.")

All four train/test files saved successfully.


In [30]:
print("\nFiles in train_test_data:")

for file in sorted(os.listdir(TRAIN_TEST_DATA_PATH)):
    print(file)


Files in train_test_data:
elliptic_test.csv
elliptic_train.csv
ethereum_test.csv
ethereum_train.csv
